# Build the reporting views

Generated from `fabric/05-gold.yaml` — do not edit by hand.

Runs **after every dimension and fact**, because these views select from tables that Spark creates through the warehouse connector. They cannot exist before the first load, which is why they are built here rather than by a deploy-time migration.

`CREATE OR ALTER`, so a re-run converges instead of failing.

In [ ]:
WAREHOUSE = 'wh_gold'
REPORTING_SCHEMA = 'bi'
PHYSICAL_SCHEMA = 'dbo'

VIEWS = ['vw_sales_summary',
 'vw_customer_360',
 'vw_product_performance',
 'vw_data_quality']
STATEMENTS = ['CREATE OR ALTER VIEW [bi].[vw_sales_summary] AS\n'
 'SELECT\n'
 '    d.full_date,\n'
 '    d.year,\n'
 '    d.quarter,\n'
 '    d.month_name,\n'
 '    d.fiscal_year,\n'
 '    c.region,\n'
 '    COUNT(DISTINCT f.order_id)  AS orders,\n'
 '    SUM(f.quantity)             AS units_sold,\n'
 '    SUM(f.recognised_revenue)   AS revenue,\n'
 '    SUM(f.gross_margin)         AS gross_margin,\n'
 '    CASE WHEN COUNT(DISTINCT f.order_id) = 0 THEN 0\n'
 '         ELSE SUM(f.recognised_revenue) / COUNT(DISTINCT f.order_id)\n'
 '    END                         AS avg_order_value\n'
 'FROM dbo.fct_sales      f\n'
 'JOIN dbo.dim_date       d ON f.order_date_sk = d.date_sk\n'
 'JOIN dbo.dim_customer   c ON f.customer_sk   = c.customer_sk\n'
 'GROUP BY d.full_date, d.year, d.quarter, d.month_name, d.fiscal_year, c.region',
 'CREATE OR ALTER VIEW [bi].[vw_customer_360] AS\n'
 'WITH customer_totals AS (\n'
 '    SELECT\n'
 '        c.customer_sk,\n'
 '        c.customer_id,\n'
 '        c.full_name,\n'
 '        c.region,\n'
 '        c.tenure_band,\n'
 '        COUNT(DISTINCT f.order_id) AS total_orders,\n'
 '        SUM(f.recognised_revenue)  AS lifetime_value,\n'
 '        MIN(d.full_date)           AS first_order_date,\n'
 '        MAX(d.full_date)           AS last_order_date\n'
 '    FROM dbo.dim_customer c\n'
 '    JOIN dbo.fct_sales    f ON f.customer_sk   = c.customer_sk\n'
 '    JOIN dbo.dim_date     d ON f.order_date_sk = d.date_sk\n'
 '    WHERE c.is_current = 1\n'
 '    GROUP BY c.customer_sk, c.customer_id, c.full_name, c.region, c.tenure_band\n'
 ')\n'
 'SELECT\n'
 '    *,\n'
 '    CASE WHEN total_orders = 0 THEN 0\n'
 '         ELSE lifetime_value / total_orders END AS avg_order_value,\n'
 '    DATEDIFF(day, last_order_date, CAST(GETDATE() AS DATE)) AS days_since_last_order,\n'
 '    -- Value tiers are quantile-based rather than fixed thresholds, so\n'
 '    -- they stay meaningful as absolute revenue grows.\n'
 '    -- NTILE rather than PERCENTILE_CONT ... OVER (), which the Fabric\n'
 '    -- Warehouse does not reliably support. Same four bands, plain window\n'
 '    -- syntax, and it stays quantile-based so the tiers remain meaningful\n'
 '    -- as absolute revenue grows.\n'
 '    CASE NTILE(10) OVER (ORDER BY lifetime_value)\n'
 "        WHEN 10 THEN 'Platinum'\n"
 "        WHEN 9  THEN 'Gold'\n"
 "        WHEN 8  THEN 'Gold'\n"
 "        WHEN 7  THEN 'Silver'\n"
 "        WHEN 6  THEN 'Silver'\n"
 "        WHEN 5  THEN 'Silver'\n"
 "        ELSE 'Bronze'\n"
 '    END AS value_tier,\n'
 '    CASE\n'
 "        WHEN DATEDIFF(day, last_order_date, CAST(GETDATE() AS DATE)) > 180 THEN 'Churn risk'\n"
 "        WHEN DATEDIFF(day, last_order_date, CAST(GETDATE() AS DATE)) >  90 THEN 'Cooling'\n"
 "        ELSE 'Active'\n"
 '    END AS engagement_status\n'
 'FROM customer_totals',
 'CREATE OR ALTER VIEW [bi].[vw_product_performance] AS\n'
 'SELECT\n'
 '    p.product_sk,\n'
 '    p.product_id,\n'
 '    p.product_name,\n'
 '    p.category,\n'
 '    p.price_band,\n'
 '    p.stock_status,\n'
 '    COUNT(DISTINCT f.order_id) AS orders,\n'
 '    SUM(f.quantity)            AS units_sold,\n'
 '    SUM(f.recognised_revenue)  AS revenue,\n'
 '    SUM(f.gross_margin)        AS gross_margin,\n'
 '    CASE WHEN SUM(f.recognised_revenue) = 0 THEN 0\n'
 '         ELSE SUM(f.gross_margin) / SUM(f.recognised_revenue) * 100\n'
 '    END                        AS margin_pct\n'
 'FROM dbo.dim_product p\n'
 'JOIN dbo.fct_sales   f ON f.product_sk = p.product_sk\n'
 'WHERE p.is_current = 1\n'
 'GROUP BY p.product_sk, p.product_id, p.product_name, p.category,\n'
 '         p.price_band, p.stock_status',
 'CREATE OR ALTER VIEW [bi].[vw_data_quality] AS\n'
 'SELECT\n'
 '    load_id,\n'
 '    table_name,\n'
 '    layer,\n'
 '    rows_in,\n'
 '    rows_out,\n'
 '    rows_quarantined,\n'
 '    rows_corrected,\n'
 '    CASE WHEN rows_in = 0 THEN 0\n'
 '         ELSE CAST(rows_out AS FLOAT) / rows_in * 100\n'
 '    END AS pass_rate_pct,\n'
 '    checks_passed,\n'
 '    checks_failed,\n'
 '    processed_at\n'
 '-- Cross-item read. The DQ log is written by EVERY layer, and each\n'
 "-- notebook writes it unqualified, so it lands in that notebook's own\n"
 '-- default lakehouse rather than here. The view consolidates them.\n'
 '--\n'
 "-- Consolidating at read time is deliberate: routing every layer's log to\n"
 '-- the warehouse would couple bronze and silver to gold for nothing more\n'
 '-- than logging.\n'
 'FROM (\n'
 '    SELECT * FROM [lh_bronze].[dbo].[dq_run_log]\n'
 '    UNION ALL\n'
 '    SELECT * FROM [lh_silver].[dbo].[dq_run_log]\n'
 ') AS dq_run_log']


In [ ]:
import com.microsoft.spark.fabric  # noqa: F401  registers the connector
import requests

workspace_id = spark.conf.get('trident.workspace.id')
token = mssparkutils.credentials.getToken('https://api.fabric.microsoft.com')

# Resolved at RUN time from the workspace this notebook is in, so the
# same artefact points at dev's warehouse in dev and qa's in qa.
warehouses = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses',
    headers={'Authorization': f'Bearer {token}'}, timeout=60).json()['value']
target = next(w for w in warehouses if w['displayName'] == WAREHOUSE)
endpoint = target['properties']['connectionString']
print(f'{WAREHOUSE} -> {endpoint}')


In [ ]:
sql_token = mssparkutils.credentials.getToken('https://database.windows.net/')
jvm = spark._jvm
props = jvm.java.util.Properties()
props.setProperty('accessToken', sql_token)
props.setProperty('encrypt', 'true')
conn = jvm.java.sql.DriverManager.getConnection(
    f'jdbc:sqlserver://{endpoint}:1433;database={WAREHOUSE}', props)
conn.setAutoCommit(True)
stmt = conn.createStatement()

# Each view is applied independently. One malformed view must not stop
# the other three -- and a single aggregate error hides how many were
# actually fine.
failed = []
for name, sql in zip(VIEWS, STATEMENTS):
    try:
        stmt.execute(sql)
        print(f'  ok      {REPORTING_SCHEMA}.{name}')
    except Exception as exc:
        failed.append(name)
        print(f'  FAILED  {REPORTING_SCHEMA}.{name}: {str(exc)[:160]}')

stmt.close(); conn.close()

if failed:
    raise RuntimeError(
        f'{len(failed)} of {len(VIEWS)} view(s) could not be created: '
        + ', '.join(failed))
print(f'\nall {len(VIEWS)} reporting view(s) are current')
